In [ ]:
# === Import principali ===
import numpy as np
import matplotlib.pyplot as plt
import os
import midas.file_reader
from datetime import datetime
import wclib as wc
import json

def plot_waveform(waveform, lenw, pmt, event_number, event_time):
    import numpy as np
    
    t = np.linspace(0,lenw, lenw)
    for ipmt in range(pmt):
        plt.subplot(pmt, 1, ipmt+1)
        plt.plot(t, waveform[ipmt])
        plt.ylabel('ch{:d} [mV]'.format(ipmt))

    return

def plot_waveform_h(waveform, lenw, pmt, hmin, hmax, vmin, vmax, event_number, event_time):
    import numpy as np
    import math
    npx = math.ceil(pmt/2)
    t = np.linspace(0,lenw, lenw)
    for ipmt in range(pmt):
        plt.subplot(npx, 2, ipmt+1)
        plt.plot(t[hmin:hmax], waveform[ipmt, ][hmin:hmax])
        if vmax!=4096 or vmin!=0: 
            plt.ylim(vmin, vmax)
        plt.ylabel('ch{:d} [mV]'.format(ipmt))

    return


verbose = False
DEBUG   = False
run     = int(input("run number [int] "))
pmt_n   = 1
hmin = 0
hmax = 1024
vmin = 0
vmax =4096

outplot = True # fa i plot

path    = '/home/mazzitel/nmv-data/WC/WC25/'

mfile = wc.open_mid(run=run, path=path, cloud=False, tag='', verbose=verbose)

# esempio lettura informazioni dall'odb  #######
odb = wc.get_bor_odb(mfile)

try:
    Run_description   = odb.data['Experiment']['Run Parameters']['Run description']
    print('Run description:', Run_description)
except:
    print('WARNING: no run description')

###############################################
# lettura equipment nel file #######

hvalue=[]
tsvalue=[]
tsunix=[]
for event in mfile:
    if event.header.is_midas_internal_event():
        print("Saw a special event")
        continue

    bank_names = ", ".join(b.name for b in event.banks.values())
    event_number = event.header.serial_number
    event_time = datetime.fromtimestamp(event.header.timestamp).strftime('%Y-%m-%d %H:%M:%S')
    if verbose or event_number % 1000==0:
        print("Event # %s of type ID %s contains banks %s" % (event_number, event.header.event_id, bank_names))
        print("Received event with timestamp %s containing banks %s" % (event.header.timestamp, bank_names))
        print("%s, banks %s" % (datetime.utcfromtimestamp(event.header.timestamp).strftime('%Y-%m-%d %H:%M:%S'), bank_names))
    for bank_name, bank in event.banks.items():
        # if ('DGH0' in bank_name): # PMTs wavform 
        
        if bank_name=='HS00': 
            hd = json.loads(bytes(event.banks['HS00'].data).decode("utf-8"))
            tw = np.asarray(event.banks['DS00'].data, dtype=np.float64)
            lenw=int(hd['WAVE_ARRAY_COUNT'])
            # init new waveform
            waveform=np.zeros((pmt_n,lenw), dtype=np.float64)
            if len(tw)>0:
                waveform[0]=tw
        # if bank_name=='HS01': 
        #     hd = json.loads(bytes(event.banks['HS01'].data).decode("utf-8"))
        #     tw = np.asarray(event.banks['DS01'].data, dtype=np.float64)
        #     lenw=int(hd['WAVE_ARRAY_COUNT'])
        #     if len(tw)>0:
        #         waveform[1]=tw
        # if bank_name=='HS02': 			
        #     hd = json.loads(bytes(event.banks['HS02'].data).decode("utf-8"))
        #     tw = np.asarray(event.banks['DS02'].data, dtype=np.float64)
        #     lenw=int(hd['WAVE_ARRAY_COUNT'])
        #     if len(tw)>0:
        #         waveform[2]=tw
        # if bank_name=='HS03': 
        #     hd = json.loads(bytes(event.banks['HS03'].data).decode("utf-8"))
        #     tw = np.asarray(event.banks['DS03'].data, dtype=np.float64)
        #     lenw=int(hd['WAVE_ARRAY_COUNT'])
        #     if len(tw)>0:
        #         waveform[3]=tw
            
            hmax=lenw   
    #            print(waveform)
            #plot_waveform_h(waveform, lenw, pmt_n, hmin, hmax, vmin, vmax, event_number, event_time)
            #plt.xlim(4500,6000)
            #print(np.trapz(waveform[0][4500:6000]))
            #plt.show()
            hvalue.append(np.trapz(waveform[0]))
            tsvalue.append(hd['TRIGGER_TIME'])
            tsunix.append(event.header.timestamp)

            
    if DEBUG:        
        if event.header.serial_number == 20: # si ferm dopo i primi 20 eventi
            break

from datetime import datetime, timezone

# trigger_times: lista di stringhe ISO 8601, una per evento
# es: "2026-01-16T17:21:25.123456"
def parse_iso(ts: str) -> float:
    # Supporta anche senza frazioni; assume timezone locale se assente (meglio: usa UTC se puoi)
    dt = datetime.fromisoformat(ts)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.timestamp()

t = np.array([parse_iso(s) for s in tsvalue], dtype=float)
t.sort()

dt = np.diff(t)  # secondi tra trigger

Tmean = dt.mean()
f_avg = 1.0 / Tmean

jitter_rms = dt.std(ddof=1)          # secondi
jitter_rel = jitter_rms / Tmean      # frazione

print("Periodo medio [s]:", Tmean)
print("Frequenza media [Hz]:", f_avg)
print("Jitter RMS [s]:", jitter_rms)
print("Jitter relativo:", jitter_rel)

N = len(tsunix)
T = tsunix[-1] - tsunix[0]

f_avg = (N - 1) / T if T > 0 else float("nan")

print("Frequenza media [Hz]:", f_avg)

print('V/div: ~', 10*hd['VERTICAL_GAIN']*(hd['MAX_VALUE']-hd['MIN_VALUE'])/8)
print('DONE')